# 00. Setup verification

Smoke-tests the local infrastructure: Keycloak (`:8080`), OPA (`:8181`),
expense-service (`:8001`), and document-service (`:8002`). Run top to bottom.
Every cell prints `[PASS]` lines on success and raises on failure.

If something fails, the most likely cause is that `docker compose up -d`
hasn't fully come up yet. Give Keycloak about 30 seconds on a cold start
and re-run.

In [1]:
import base64
import json
import httpx

KEYCLOAK_URL = "http://localhost:8080"
REALM        = "agentauth"
OPA_URL      = "http://localhost:8181"
EXPENSE_URL  = "http://localhost:8001"
DOCUMENT_URL = "http://localhost:8002"

AGENT_CLIENT_ID     = "agent-client"
AGENT_CLIENT_SECRET = "agent-client-secret"

def ok(label):
    print(f"  [PASS] {label}")

def decode_jwt(token):
    parts = token.split(".")
    payload = parts[1] + "=" * (-len(parts[1]) % 4)
    return json.loads(base64.urlsafe_b64decode(payload))

## 1. Keycloak well-known endpoint reachable

In [2]:
print("checking Keycloak well-known...")
r = httpx.get(f"{KEYCLOAK_URL}/realms/{REALM}/.well-known/openid-configuration", timeout=5.0)
r.raise_for_status()
config = r.json()
assert config["issuer"].endswith(f"/realms/{REALM}"), config["issuer"]
ok(f"realm reachable, issuer = {config['issuer']}")
ok(f"token endpoint = {config['token_endpoint']}")

checking Keycloak well-known...
  [PASS] realm reachable, issuer = http://localhost:8080/realms/agentauth
  [PASS] token endpoint = http://localhost:8080/realms/agentauth/protocol/openid-connect/token


## 2. Direct grant for alice and decoded custom claims

In [3]:
def get_user_jwt(username, password="password"):
    r = httpx.post(
        f"{KEYCLOAK_URL}/realms/{REALM}/protocol/openid-connect/token",
        data={
            "client_id": AGENT_CLIENT_ID,
            "client_secret": AGENT_CLIENT_SECRET,
            "grant_type": "password",
            "username": username,
            "password": password,
            "scope": "openid",
        },
        timeout=5.0,
    )
    r.raise_for_status()
    return r.json()["access_token"]

alice_jwt = get_user_jwt("alice")
claims = decode_jwt(alice_jwt)

assert claims["preferred_username"] == "alice", claims
assert claims["role"] == "employee", claims
assert claims["department"] == "engineering", claims
assert claims["reports_to"] == "bob", claims
assert claims["azp"] == AGENT_CLIENT_ID, claims
assert "expense-service-client" in claims["aud"], claims["aud"]
assert "document-service-client" in claims["aud"], claims["aud"]

ok(f"alice token has all custom claims (role={claims['role']}, dept={claims['department']}, reports_to={claims['reports_to']})")
ok(f"alice token aud = {claims['aud']}  (note: BROAD, both services)")
ok(f"alice token azp = {claims['azp']}  (the requesting client)")

  [PASS] alice token has all custom claims (role=employee, dept=engineering, reports_to=bob)
  [PASS] alice token aud = ['expense-service-client', 'document-service-client']  (note: BROAD, both services)
  [PASS] alice token azp = agent-client  (the requesting client)


## 3. Smoke test bob and dave

In [4]:
for username in ("bob", "dave"):
    c = decode_jwt(get_user_jwt(username))
    print(f"  {username}: role={c['role']}, dept={c['department']}, reports_to={c.get('reports_to') or '<none>'}")

bob = decode_jwt(get_user_jwt("bob"))
assert bob["role"] == "manager", bob
ok("bob is manager")

dave = decode_jwt(get_user_jwt("dave"))
assert dave["role"] == "admin", dave
ok("dave is admin")

  bob: role=manager, dept=engineering, reports_to=<none>
  dave: role=admin, dept=platform, reports_to=<none>
  [PASS] bob is manager
  [PASS] dave is admin


## 4. OPA packages loaded

In [5]:
r = httpx.get(f"{OPA_URL}/v1/data/agentauth", timeout=5.0)
r.raise_for_status()
result = r.json()["result"]
assert "agent_side" in result, result
assert "tool_side" in result, result
ok("OPA agentauth.agent_side package loaded")
ok("OPA agentauth.tool_side package loaded")

  [PASS] OPA agentauth.agent_side package loaded
  [PASS] OPA agentauth.tool_side package loaded


## 5. OPA agent-side and tool-side decisions

In [6]:
def opa_decide(package, input_):
    r = httpx.post(f"{OPA_URL}/v1/data/agentauth/{package}/decision", json={"input": input_}, timeout=5.0)
    r.raise_for_status()
    return r.json()["result"]

# agent-side: alice (employee) calling get_expenses, allow
d = opa_decide("agent_side", {"user": {"role": "employee"}, "tool": "get_expenses", "action": "read"})
assert d["allow"], d
ok(f"agent_side: alice can call get_expenses ({d['reason']})")

# agent-side: alice (employee) trying approve_expense, deny
d = opa_decide("agent_side", {"user": {"role": "employee"}, "tool": "approve_expense", "action": "approve"})
assert not d["allow"], d
ok(f"agent_side: alice CANNOT call approve_expense ({d['reason']})")

# tool-side: bob approves alice's expense (manager-of-target), allow
d = opa_decide("tool_side", {"user_id": "bob", "target_user_id": "alice", "action": "approve", "resource_type": "expense"})
assert d["allow"], d
assert d["reason"] == "manager-of-target relationship", d
ok(f"tool_side: bob can approve alice's expense ({d['reason']})")

# tool-side: alice tries to approve bob's expense, deny
d = opa_decide("tool_side", {"user_id": "alice", "target_user_id": "bob", "action": "approve", "resource_type": "expense"})
assert not d["allow"], d
ok(f"tool_side: alice CANNOT approve bob's expense ({d['reason']})")

# tool-side: dave (admin) reads anything, allow with admin override
d = opa_decide("tool_side", {"user_id": "dave", "target_user_id": "alice", "action": "read", "resource_type": "expense"})
assert d["allow"] and d["reason"] == "admin override", d
ok(f"tool_side: dave (admin) can read anything ({d['reason']})")

  [PASS] agent_side: alice can call get_expenses (permitted)
  [PASS] agent_side: alice CANNOT call approve_expense (role not permitted to invoke this tool)
  [PASS] tool_side: bob can approve alice's expense (manager-of-target relationship)
  [PASS] tool_side: alice CANNOT approve bob's expense (no rule permits this action on this target)


  [PASS] tool_side: dave (admin) can read anything (admin override)


## 6. Backend services healthy

In [7]:
for label, url in (("expense-service", EXPENSE_URL), ("document-service", DOCUMENT_URL)):
    r = httpx.get(f"{url}/healthz", timeout=5.0)
    r.raise_for_status()
    body = r.json()
    assert body["status"] == "ok", body
    ok(f"{label}: {body}")

  [PASS] expense-service: {'status': 'ok', 'service': 'expense-service', 'tool_side_authz': False}
  [PASS] document-service: {'status': 'ok', 'service': 'document-service'}


## 7. /debug/last-request endpoints

In [8]:
for label, url in (("expense-service", EXPENSE_URL), ("document-service", DOCUMENT_URL)):
    r = httpx.get(f"{url}/debug/last-request", timeout=5.0)
    r.raise_for_status()
    print(f"  {label} last-request: method={r.json().get('method')}")
ok("/debug/last-request reachable on both services")

  expense-service last-request: method=jwt


  document-service last-request: method=string_id
  [PASS] /debug/last-request reachable on both services


## 8. Standard Token Exchange v2 round-trip

agent-client exchanges alice's broad user JWT for a token whose `aud` is
just `expense-service-client`. The exchanged token still says `sub=alice`,
and `azp` identifies the agent as the requester.

In [9]:
exchange_resp = httpx.post(
    f"{KEYCLOAK_URL}/realms/{REALM}/protocol/openid-connect/token",
    data={
        "client_id":           AGENT_CLIENT_ID,
        "client_secret":       AGENT_CLIENT_SECRET,
        "grant_type":          "urn:ietf:params:oauth:grant-type:token-exchange",
        "subject_token":       alice_jwt,
        "subject_token_type":  "urn:ietf:params:oauth:token-type:access_token",
        "requested_token_type":"urn:ietf:params:oauth:token-type:access_token",
        "audience":            "expense-service-client",
    },
    timeout=5.0,
)
exchange_resp.raise_for_status()
exchanged_token = exchange_resp.json()["access_token"]
exchanged_claims = decode_jwt(exchanged_token)

assert exchanged_claims["aud"] == "expense-service-client", exchanged_claims["aud"]
ok(f"aud NARROWED: {exchanged_claims['aud']} (was {claims['aud']})")

assert exchanged_claims["azp"] == AGENT_CLIENT_ID, exchanged_claims
ok(f"azp == {AGENT_CLIENT_ID} (audit trail of who requested the exchange)")

assert exchanged_claims["preferred_username"] == "alice", exchanged_claims
ok(f"preferred_username == alice (subject preserved)")

assert exchanged_claims["sub"] == claims["sub"], (exchanged_claims["sub"], claims["sub"])
ok(f"sub == {exchanged_claims['sub'][:8]}... (UUID preserved across exchange)")

  [PASS] aud NARROWED: expense-service-client (was ['expense-service-client', 'document-service-client'])
  [PASS] azp == agent-client (audit trail of who requested the exchange)
  [PASS] preferred_username == alice (subject preserved)
  [PASS] sub == dbee663c... (UUID preserved across exchange)


## 9. Send the exchanged token to expense-service and confirm `scoped_jwt` detection

In [10]:
r = httpx.get(
    f"{EXPENSE_URL}/expenses",
    headers={"Authorization": f"Bearer {exchanged_token}"},
    timeout=5.0,
)
r.raise_for_status()
body = r.json()
assert body["identity_method"] == "scoped_jwt", body
ok(f"expense-service classified the exchanged token as scoped_jwt (count={body['count']})")

# And the broad user JWT should be classified as plain `jwt`
r = httpx.get(
    f"{EXPENSE_URL}/expenses",
    headers={"Authorization": f"Bearer {alice_jwt}"},
    timeout=5.0,
)
body = r.json()
assert body["identity_method"] == "jwt", body
ok(f"expense-service classified the BROAD user JWT as jwt (count={body['count']})")

  [PASS] expense-service classified the exchanged token as scoped_jwt (count=4)
  [PASS] expense-service classified the BROAD user JWT as jwt (count=4)


## All checks passed

Everything above ended in `[PASS]` lines and no exceptions, so the local
infrastructure is working: realm imported with all custom claims, OPA
policies loaded for both packages, both backend services running, and
Standard Token Exchange v2 producing the expected narrowed-audience tokens.

You can move on to notebook 01.